# PaceAlgo ML — Notebook 12: Model Battery

Vergleich von 4 Modellarchitekturen auf der **besten Feature-Konfiguration aus NB 11** (`phase1_best_config.json`):

1. **LightGBM** (Pine-budget: 30 trees, depth 3) — bisheriges baseline
2. **XGBoost** (gleiche depth + trees) — kann marginal andere Splits finden
3. **CatBoost** (gleiche) — bessere categorical handling, oft robusterer Default
4. **Voting Ensemble** — Mittelwert der 3 calibrated probabilities

**Strikte Methodik:**
- Identische Train/Val/Test-Splits (Walk-Forward)
- Identische Tier-Cutoffs aus VAL-Set abgeleitet
- Pro Modell × Tier: PF / WR / Expectancy / Stability-CV
- Pine-Export-Eignung berücksichtigt (LGBM/XGB exportierbar; CatBoost komplex)

**Entscheidungslogik:**
- Bestes Modell muss `Premium-Tier-PF > NB-11-Bestwert + 0.05` haben um den Wechsel zu rechtfertigen
- Sonst bleibt LightGBM (einfachster Pine-Export)

## 1. Setup

In [ ]:
import sys, os
from pathlib import Path

IS_COLAB = 'google.colab' in sys.modules
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/pace-algo'
    DATA_PROCESSED = Path('/content/processed')
    DATA_EXT = Path('/content/processed_v2')
    ARTIFACTS = Path(PROJECT_ROOT) / 'artifacts'
    REPORTS_DIR = ARTIFACTS / 'reports'
    MODELS_DIR = ARTIFACTS / 'models'
    DRIVE_BACKUP = Path(PROJECT_ROOT) / 'data_cache' / 'processed'
    os.chdir(PROJECT_ROOT)
    !rm -rf /tmp/pace-algo
    !git clone -q https://github.com/ecoNC/pace-algo.git /tmp/pace-algo
    !cp -rf /tmp/pace-algo/core/* {PROJECT_ROOT}/core/
    print('Core code updated from GitHub')
    DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

    # CRITICAL: restore labels + base features from Drive if local is empty
    # (these are needed when NB 12 rebuilds extended features in Section 2.5)
    n_local_labels = len(list(DATA_PROCESSED.glob('labels_*.parquet')))
    n_drive_labels = len(list(DRIVE_BACKUP.glob('labels_*.parquet'))) if DRIVE_BACKUP.exists() else 0
    print(f'Labels in local: {n_local_labels}, Drive backup: {n_drive_labels}')
    if n_local_labels < n_drive_labels:
        print('Restoring labels + base features from Drive...')
        !rsync -ah {DRIVE_BACKUP}/ {DATA_PROCESSED}/
        n_local_labels = len(list(DATA_PROCESSED.glob('labels_*.parquet')))
        print(f'After restore: {n_local_labels} label files in /content/processed/')
else:
    PROJECT_ROOT = os.path.abspath('..')
    os.chdir(PROJECT_ROOT)
    from core.config import DATA_PROCESSED, ARTIFACTS_REPORTS, ARTIFACTS_MODELS
    REPORTS_DIR = ARTIFACTS_REPORTS
    MODELS_DIR = ARTIFACTS_MODELS
    DATA_EXT = DATA_PROCESSED.parent / 'processed_v2'

MODELS_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, PROJECT_ROOT)
for mod in list(sys.modules.keys()):
    if mod.startswith('core'):
        del sys.modules[mod]


In [ ]:
!pip install -q lightgbm xgboost catboost 2>&1 | tail -1

import json
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score
from tqdm.notebook import tqdm

from core.config import (
    FX_SYMBOLS, METAL_SYMBOLS, PRIMARY_TIMEFRAMES, DEV_HOLDOUT_SYMBOLS,
    TRAIN_END, VAL_END, HTF_CONTEXT_TIMEFRAMES,
)
from core.features import (
    compute_features, attach_macro, attach_htf_context,
    compute_smc_features, compute_session_features, compute_htf_interactions,
)
from core.features.engineer import atr as atr_fn
from core.train import walk_forward_split, binary_label_for_long

# Path setup — match NB 11
if IS_COLAB:
    DATA_RAW = Path(PROJECT_ROOT) / 'data_cache' / 'raw'
else:
    from core.config import DATA_RAW

R_VALUE = 1.5
WIN_R = 1.5
LOSS_R = 1.0
print('Libraries loaded:')
print(f'  LightGBM: {lgb.__version__}')
print(f'  XGBoost:  {xgb.__version__}')
import catboost
print(f'  CatBoost: {catboost.__version__}')

# Make sure extended directory exists
DATA_EXT.mkdir(parents=True, exist_ok=True)
print(f'\nExtended features dir: {DATA_EXT}')

# Check if extended files exist (may have been wiped between NB 11 and NB 12)
n_existing = len(list(DATA_EXT.glob('*_extended.parquet')))
print(f'Extended feature files present: {n_existing}')

# Drive backup location (in case we sync later)
if IS_COLAB:
    EXT_DRIVE_BACKUP = Path(PROJECT_ROOT) / 'data_cache' / 'processed_v2'
    EXT_DRIVE_BACKUP.mkdir(parents=True, exist_ok=True)
    n_backup = len(list(EXT_DRIVE_BACKUP.glob('*_extended.parquet')))
    print(f'Drive backup files: {n_backup}')
    if n_existing < n_backup:
        print('Restoring extended features from Drive backup...')
        !rsync -ah {EXT_DRIVE_BACKUP}/ {DATA_EXT}/
        n_existing = len(list(DATA_EXT.glob('*_extended.parquet')))
        print(f'After restore: {n_existing} files')


## 2. Load Best Config from NB 11

In [ ]:
# NB 11 best config — saved as artifacts/reports/phase1_best_config.json
# Fallback: hardcoded from NB 11 winning configuration (FX-only, 27 features)
HARDCODED_FALLBACK = {
    'experiment_name': 'asset_FX_only',
    'asset_scope':     'FX_only',
    'feature_cols': [
        'hour_sin', 'dist_to_swing_low_atr', 'htf_1h_rsi_14', 'realized_vol_20',
        'atr_percentile_100', 'atr_pct', 'dist_to_swing_high_atr', 'volume_z_score',
        'ema_20_slope_atr', 'hour_cos', 'momentum_composite', 'rvol_20',
        'adx_14', 'ema_20_dist_atr', 'htf_1h_atr_percentile_100',
        'htf_ltf_agree_bull', 'htf_ltf_agree_bear', 'htf_ltf_counter_trend',
        'htf_ltf_alignment_score', 'ltf_rsi_minus_htf_rsi',
        'both_rsi_oversold', 'both_rsi_overbought', 'vol_pct_diff_htf',
        'both_high_vol', 'both_low_vol', 'pullback_in_bull', 'pullback_in_bear',
    ],
    'n_features': 27,
    'premium_pf_oos': 2.015,
    'premium_wr_oos': 0.573,
    'premium_expR_oos': 0.4264,
}

config_path = REPORTS_DIR / 'phase1_best_config.json'
if config_path.exists():
    with open(config_path) as f:
        best_config = json.load(f)
    print(f'Loaded best config from {config_path}')
else:
    best_config = HARDCODED_FALLBACK
    print(f'phase1_best_config.json not found — using HARDCODED fallback (NB 11 winning config)')

print(f'\n  experiment_name: {best_config["experiment_name"]}')
print(f'  asset_scope:     {best_config["asset_scope"]}')
print(f'  n_features:      {best_config["n_features"]}')
print(f'  PF (NB11):       {best_config["premium_pf_oos"]:.4f}')
print(f'  WR (NB11):       {best_config["premium_wr_oos"]*100:.1f}%')
print(f'  ExpR (NB11):     {best_config["premium_expR_oos"]:+.4f}')
print(f'  features: {best_config["feature_cols"][:5]} ... (+{len(best_config["feature_cols"])-5} more)')


## 2.5 Rebuild Extended Features if Missing

NB 11 baut diese in `/content/processed_v2/` — Colab löscht /content/ bei Runtime-Wechsel.
Diese Cell baut sie aus `data_cache/raw/` neu auf, falls sie fehlen.


In [ ]:
def load_ohlcv_raw(symbol, tf):
    p = DATA_RAW / f'{symbol}_{tf}.parquet'
    return pd.read_parquet(p) if p.exists() else None

# Validate: do existing extended files have the 'label' column?
# If not, they were built without labels (NB 11 didn't have labels available) — delete and rebuild
def has_label_column(path):
    try:
        # Read only first few rows to check schema
        df_check = pd.read_parquet(path, columns=None).head(1)
        return 'label' in df_check.columns
    except Exception:
        return False

existing_files = list(DATA_EXT.glob('*_extended.parquet'))
if existing_files:
    sample_path = existing_files[0]
    if not has_label_column(sample_path):
        print(f'WARNING: existing extended files missing \"label\" column.')
        print(f'  Deleting {len(existing_files)} stale files to force rebuild with labels...')
        for f in existing_files:
            f.unlink()
        # Also clear Drive backup so it doesn't restore broken files
        if IS_COLAB and EXT_DRIVE_BACKUP.exists():
            for f in EXT_DRIVE_BACKUP.glob('*_extended.parquet'):
                f.unlink()
        print('  Stale files cleared.')

# Check what's needed
ALL_TRAIN_SYMBOLS = FX_SYMBOLS + METAL_SYMBOLS  # incl. GBPUSD even though dev-holdout
expected_files = {f'{s}_{tf}_extended.parquet' for s in ALL_TRAIN_SYMBOLS for tf in PRIMARY_TIMEFRAMES}
present_files = {p.name for p in DATA_EXT.glob('*_extended.parquet')}
missing = expected_files - present_files
print(f'Expected: {len(expected_files)} files, present: {len(present_files)}, missing: {len(missing)}')

if missing:
    print(f'\nBuilding missing extended features (this takes ~10-15 min)...')
    macro_path = DATA_RAW / 'macro_daily.parquet'
    macro = pd.read_parquet(macro_path) if macro_path.exists() else pd.DataFrame()

    n_label_files = len(list(DATA_PROCESSED.glob('labels_*.parquet')))
    if n_label_files == 0:
        print('  WARNING: no label files in DATA_PROCESSED. Extended features will be built WITHOUT labels.')
        print('  This will cause downstream failures. Run NB 04 first to generate labels.')

    for symbol in tqdm(ALL_TRAIN_SYMBOLS, desc='Symbols'):
        htf_feats = {}
        for htf in HTF_CONTEXT_TIMEFRAMES:
            d = load_ohlcv_raw(symbol, htf)
            htf_feats[htf] = compute_features(d) if (d is not None and not d.empty) else pd.DataFrame()

        for tf in PRIMARY_TIMEFRAMES:
            out_path = DATA_EXT / f'{symbol}_{tf}_extended.parquet'
            if out_path.exists():
                continue
            ohlcv = load_ohlcv_raw(symbol, tf)
            if ohlcv is None or ohlcv.empty:
                continue
            base = compute_features(ohlcv)
            base = attach_htf_context(base, htf_feats.get('1h', pd.DataFrame()), htf_feats.get('4h', pd.DataFrame()))
            base = attach_macro(base, macro)
            atr14 = atr_fn(ohlcv['high'], ohlcv['low'], ohlcv['close'], 14).values
            ema_align = base['ema_alignment'].fillna(0).values
            smc = compute_smc_features(ohlcv, atr14, ema_align)
            sess = compute_session_features(ohlcv, atr14)
            inter = compute_htf_interactions(base)
            ext = pd.concat([base, smc, sess, inter], axis=1)
            # Attach labels (from R=1.5 labeling) — MUST be present
            label_path = DATA_PROCESSED / f'labels_{symbol}_{tf}_R{R_VALUE}.parquet'
            if not label_path.exists():
                print(f'    SKIP {symbol} {tf}: labels missing at {label_path}')
                continue
            labels = pd.read_parquet(label_path)
            ext = ext.join(labels[['label']], how='inner')
            ext['symbol'] = symbol
            ext['timeframe'] = tf
            ext.to_parquet(out_path, compression='zstd')
        htf_feats.clear()

    n_now = len(list(DATA_EXT.glob('*_extended.parquet')))
    print(f'\nExtended features now ready: {n_now} files')
else:
    print('All extended features already present — skipping rebuild.')

# Sync to Drive backup so we don't have to rebuild next session
if IS_COLAB:
    print('\nSyncing extended features to Drive backup...')
    !rsync -ah {DATA_EXT}/ {EXT_DRIVE_BACKUP}/
    print(f'Drive backup updated.')


## 3. Load Extended Dataset (same as NB 11)

In [ ]:
def load_ext(sym, tf):
    p = DATA_EXT / f'{sym}_{tf}_extended.parquet'
    return pd.read_parquet(p) if p.exists() else None

scope = best_config['asset_scope']
if scope == 'FX_only':
    symbols = FX_SYMBOLS
elif scope == 'Gold_only':
    symbols = METAL_SYMBOLS
else:
    symbols = FX_SYMBOLS + METAL_SYMBOLS

drop = set(DEV_HOLDOUT_SYMBOLS)
frames = []
for s in symbols:
    if s in drop: continue
    for tf in PRIMARY_TIMEFRAMES:
        d = load_ext(s, tf)
        if d is not None and not d.empty:
            frames.append(d)
df_full = pd.concat(frames, axis=0).sort_index() if frames else pd.DataFrame()
print(f'Stacked dataset for scope "{scope}": {df_full.shape}')

features = best_config['feature_cols']
available = [f for f in features if f in df_full.columns]
print(f'Available features: {len(available)} of {len(features)}')

df_c = df_full.dropna(subset=available + ['label'])
train_df, val_df, test_df = walk_forward_split(df_c, TRAIN_END, VAL_END)
print(f'train: {len(train_df):,}  val: {len(val_df):,}  test: {len(test_df):,}')

X_tr = train_df[available].values; y_tr = binary_label_for_long(train_df['label']).values
X_vl = val_df[available].values; y_vl = binary_label_for_long(val_df['label']).values
X_ts = test_df[available].values; y_ts = binary_label_for_long(test_df['label']).values

print(f'\nClass balance — train: {y_tr.mean():.3f}, val: {y_vl.mean():.3f}, test: {y_ts.mean():.3f}')

## 4. Train 3 Models with Equivalent Hyperparameters

In [ ]:
# Common hyperparams: 30 trees, depth 3, mild L2 reg
models = {}

print('Training LightGBM...')
lgb_params = {
    'objective': 'binary', 'metric': 'binary_logloss',
    'num_leaves': 7, 'max_depth': 3, 'min_data_in_leaf': 200,
    'learning_rate': 0.05, 'num_iterations': 30, 'lambda_l2': 0.5,
    'feature_fraction': 0.85, 'bagging_fraction': 0.85, 'bagging_freq': 5,
    'is_unbalance': True, 'verbose': -1, 'n_jobs': -1,
}
td = lgb.Dataset(X_tr, label=y_tr)
vd = lgb.Dataset(X_vl, label=y_vl, reference=td)
models['LightGBM'] = lgb.train(lgb_params, td, num_boost_round=30,
                                  valid_sets=[vd], valid_names=['val'],
                                  callbacks=[lgb.log_evaluation(period=0)])
print('  LightGBM done')

In [ ]:
print('Training XGBoost...')
pos_weight = (1 - y_tr.mean()) / y_tr.mean()  # equivalent to is_unbalance
xgb_clf = xgb.XGBClassifier(
    n_estimators=30,
    max_depth=3,
    learning_rate=0.05,
    reg_lambda=0.5,
    subsample=0.85,
    colsample_bytree=0.85,
    scale_pos_weight=pos_weight,
    use_label_encoder=False,
    eval_metric='logloss',
    n_jobs=-1,
    verbosity=0,
)
xgb_clf.fit(X_tr, y_tr, eval_set=[(X_vl, y_vl)], verbose=False)
models['XGBoost'] = xgb_clf
print('  XGBoost done')

In [ ]:
print('Training CatBoost...')
cat_clf = CatBoostClassifier(
    iterations=30,
    depth=3,
    learning_rate=0.05,
    l2_leaf_reg=3.0,
    rsm=0.85,
    auto_class_weights='Balanced',
    verbose=False,
    thread_count=-1,
)
cat_clf.fit(X_tr, y_tr, eval_set=(X_vl, y_vl), use_best_model=False)
models['CatBoost'] = cat_clf
print('  CatBoost done')

print('\nAll 3 base models trained.')

## 5. Predict + Voting Ensemble

In [ ]:
val_probas = {}
test_probas = {}

val_probas['LightGBM'] = models['LightGBM'].predict(X_vl)
test_probas['LightGBM'] = models['LightGBM'].predict(X_ts)

val_probas['XGBoost'] = models['XGBoost'].predict_proba(X_vl)[:, 1]
test_probas['XGBoost'] = models['XGBoost'].predict_proba(X_ts)[:, 1]

val_probas['CatBoost'] = models['CatBoost'].predict_proba(X_vl)[:, 1]
test_probas['CatBoost'] = models['CatBoost'].predict_proba(X_ts)[:, 1]

# Voting ensemble = average
val_probas['Voting'] = (val_probas['LightGBM'] + val_probas['XGBoost'] + val_probas['CatBoost']) / 3
test_probas['Voting'] = (test_probas['LightGBM'] + test_probas['XGBoost'] + test_probas['CatBoost']) / 3

print('All probas computed.')
for name in val_probas:
    print(f'  {name}: VAL range [{val_probas[name].min():.3f}, {val_probas[name].max():.3f}]')

## 6. Per-Model × Per-Tier Evaluation

In [ ]:
def tier_metrics(labels):
    wins = int((labels == 1).sum())
    losses = int((labels == -1).sum())
    neutrals = int((labels == 0).sum())
    total = wins + losses + neutrals
    pf = (wins * WIN_R) / (losses * LOSS_R) if losses > 0 else (float('inf') if wins > 0 else 0.0)
    wr = wins / (wins + losses) if (wins + losses) > 0 else 0.0
    expR = (wins * WIN_R - losses * LOSS_R) / total if total > 0 else 0.0
    return {'n': total, 'wins': wins, 'losses': losses, 'wr': wr, 'pf': pf, 'expR': expR}

rows = []
for model_name in ['LightGBM', 'XGBoost', 'CatBoost', 'Voting']:
    vp = val_probas[model_name]
    tp = test_probas[model_name]
    vs = np.sort(vp)[::-1]
    cutoff_std = float(vs[max(1, int(len(vs) * 0.10) - 1)])
    cutoff_high = float(vs[max(1, int(len(vs) * 0.03) - 1)])
    cutoff_prem = float(vs[max(1, int(len(vs) * 0.01) - 1)])
    # AUC on TEST for reference
    try:
        test_auc = float(roc_auc_score(y_ts, tp))
    except Exception:
        test_auc = float('nan')
    for tier_name, cutoff in [('Standard', cutoff_std), ('High', cutoff_high), ('Premium', cutoff_prem)]:
        mask = tp >= cutoff
        sub_labels = test_df['label'].iloc[mask.nonzero()[0]]
        m = tier_metrics(sub_labels)
        m['model'] = model_name
        m['tier'] = tier_name
        m['cutoff'] = cutoff
        m['auc'] = test_auc
        rows.append(m)

battery_df = pd.DataFrame(rows)

print('=== Profit Factor by model × tier ===')
display(battery_df.pivot_table(index='model', columns='tier', values='pf').round(4)[['Standard', 'High', 'Premium']])

print('\n=== Win Rate ===')
display(battery_df.pivot_table(index='model', columns='tier', values='wr').round(4)[['Standard', 'High', 'Premium']])

print('\n=== Expectancy per Trade ===')
display(battery_df.pivot_table(index='model', columns='tier', values='expR').round(4)[['Standard', 'High', 'Premium']])

print('\n=== Trade Count ===')
display(battery_df.pivot_table(index='model', columns='tier', values='n').round(0)[['Standard', 'High', 'Premium']])

print('\n=== TEST AUC per model ===')
display(battery_df.drop_duplicates('model')[['model', 'auc']].set_index('model').round(4))

## 7. Per-Year Stability per Model (Premium Tier)

In [ ]:
test_df_copy = test_df.copy()
test_df_copy['year'] = test_df_copy.index.year

stability_rows = []
for model_name in ['LightGBM', 'XGBoost', 'CatBoost', 'Voting']:
    vp = val_probas[model_name]
    tp = test_probas[model_name]
    vs = np.sort(vp)[::-1]
    cutoff_prem = float(vs[max(1, int(len(vs) * 0.01) - 1)])
    test_df_copy['proba'] = tp
    yearly = {}
    for year, sub in test_df_copy.groupby('year'):
        mask = sub['proba'] >= cutoff_prem
        sub_l = sub.loc[mask, 'label']
        if len(sub_l) < 30:
            continue
        m = tier_metrics(sub_l)
        yearly[int(year)] = m
    pf_values = [m['pf'] for m in yearly.values() if not np.isinf(m['pf']) and m['pf'] > 0]
    if len(pf_values) >= 2:
        cv = float(np.std(pf_values) / np.mean(pf_values))
    else:
        cv = float('nan')
    for year, m in yearly.items():
        stability_rows.append({'model': model_name, 'year': year, 'pf': m['pf'], 'wr': m['wr'], 'n': m['n']})
    print(f'{model_name}: years tested = {sorted(yearly.keys())}, stability CV = {cv:.3f}')
    print(f'  yearly PFs: {[(y, round(m["pf"], 3)) for y, m in yearly.items()]}')

stab_df = pd.DataFrame(stability_rows)
if not stab_df.empty:
    print('\nPremium PF per year per model:')
    display(stab_df.pivot_table(index='year', columns='model', values='pf').round(4))

## 7.5 Trades pro Tag pro Modell × Tier

User-facing-relevant: wie viele Signale liefert jedes Modell pro Tag?


In [ ]:
test_days = (test_df.index.max() - test_df.index.min()).days
n_test_symbols = test_df['symbol'].nunique() if 'symbol' in test_df.columns else 1
print(f'TEST period: {test_days} days across {n_test_symbols} symbol(s)')

trades_per_day_rows = []
for r in rows:
    per_day_total = r['n'] / max(test_days, 1)
    per_day_per_symbol = per_day_total / max(n_test_symbols, 1)
    trades_per_day_rows.append({
        'model': r['model'],
        'tier': r['tier'],
        'total_trades': r['n'],
        'trades_per_day_all_symbols': per_day_total,
        'trades_per_day_per_symbol': per_day_per_symbol,
    })
tpd_df = pd.DataFrame(trades_per_day_rows)
print('\n=== Trades per Day per Tier per Model (TEST set, all symbols combined) ===')
display(tpd_df.pivot_table(index='model', columns='tier', values='trades_per_day_all_symbols')
        .round(2)[['Standard', 'High', 'Premium']])

print('\n=== Trades per Day PER SYMBOL ===')
display(tpd_df.pivot_table(index='model', columns='tier', values='trades_per_day_per_symbol')
        .round(2)[['Standard', 'High', 'Premium']])


## 7.7 GBPUSD Hold-Out Test pro Modell

**Wichtigster Test:** GBPUSD wurde im Training nie gesehen. Wenn die Edge generalisierungsfähig ist, sollte sie auch hier funktionieren.


In [ ]:
# Load GBPUSD extended features (held-out symbol)
ho_frames = []
for tf in PRIMARY_TIMEFRAMES:
    d = load_ext('GBPUSD', tf)
    if d is not None and not d.empty:
        ho_frames.append(d)
ho_df = pd.concat(ho_frames, axis=0).sort_index() if ho_frames else pd.DataFrame()

# Filter to OOS window only (>= VAL_END)
val_end_ts = pd.Timestamp(VAL_END)
if val_end_ts.tz is None:
    val_end_ts = val_end_ts.tz_localize('UTC')
else:
    val_end_ts = val_end_ts.tz_convert('UTC')

ho_df = ho_df[ho_df.index >= val_end_ts]
ho_df = ho_df.dropna(subset=available + ['label'])
print(f'GBPUSD hold-out OOS: {len(ho_df):,} bars')

ho_rows = []
ho_test_probas = {}
for model_name in ['LightGBM', 'XGBoost', 'CatBoost', 'Voting']:
    if model_name == 'LightGBM':
        ho_p = models['LightGBM'].predict(ho_df[available].values)
    elif model_name == 'Voting':
        ho_p = (models['LightGBM'].predict(ho_df[available].values) +
                 models['XGBoost'].predict_proba(ho_df[available].values)[:, 1] +
                 models['CatBoost'].predict_proba(ho_df[available].values)[:, 1]) / 3
    else:
        ho_p = models[model_name].predict_proba(ho_df[available].values)[:, 1]
    ho_test_probas[model_name] = ho_p

    # Use same VAL-derived cutoffs as the in-sample test (transferred)
    vp = val_probas[model_name]
    vs = np.sort(vp)[::-1]
    cutoff_std = float(vs[max(1, int(len(vs) * 0.10) - 1)])
    cutoff_high = float(vs[max(1, int(len(vs) * 0.03) - 1)])
    cutoff_prem = float(vs[max(1, int(len(vs) * 0.01) - 1)])

    for tier_name, cutoff in [('Standard', cutoff_std), ('High', cutoff_high), ('Premium', cutoff_prem)]:
        mask = ho_p >= cutoff
        sub_labels = ho_df['label'].iloc[mask.nonzero()[0]]
        m = tier_metrics(sub_labels)
        m['model'] = model_name
        m['tier'] = tier_name
        ho_rows.append(m)

ho_battery = pd.DataFrame(ho_rows)

print('\n=== GBPUSD Hold-Out Profit Factor ===')
display(ho_battery.pivot_table(index='model', columns='tier', values='pf')
        .round(4)[['Standard', 'High', 'Premium']])

print('\n=== GBPUSD Hold-Out Win Rate ===')
display(ho_battery.pivot_table(index='model', columns='tier', values='wr')
        .round(4)[['Standard', 'High', 'Premium']])

print('\n=== GBPUSD Hold-Out Trade Count ===')
display(ho_battery.pivot_table(index='model', columns='tier', values='n')
        .round(0)[['Standard', 'High', 'Premium']])


## 7.9 Consensus / Meta-Labeling Filter

**Idee:** Statt Voting (Mittelwert) als 4. Modell, teste Consensus: Premium-Signal **nur** wenn alle 3 Modelle (LGBM + XGB + CatBoost) zustimmen. Das ist Meta-Labeling auf Modell-Ebene: ein Modell sagt "Trade", die anderen filtern.

Wenn der Consensus-PF deutlich besser als das beste Einzelmodell ist, könnten wir es als Filter-Layer einbauen (auch wenn nur das beste Einzelmodell in Pine läuft).


In [ ]:
def per_model_cutoff(vp, pct):
    vs = np.sort(vp)[::-1]
    return float(vs[max(1, int(len(vs) * pct / 100) - 1)])

cutoffs_per_model = {}
for m in ['LightGBM', 'XGBoost', 'CatBoost']:
    cutoffs_per_model[m] = {
        'Standard': per_model_cutoff(val_probas[m], 10),
        'High':     per_model_cutoff(val_probas[m], 3),
        'Premium':  per_model_cutoff(val_probas[m], 1),
    }

# Consensus on in-sample TEST
consensus_rows = []
for tier_name in ['Standard', 'High', 'Premium']:
    mask = (
        (test_probas['LightGBM'] >= cutoffs_per_model['LightGBM'][tier_name]) &
        (test_probas['XGBoost']  >= cutoffs_per_model['XGBoost'][tier_name]) &
        (test_probas['CatBoost'] >= cutoffs_per_model['CatBoost'][tier_name])
    )
    sub_labels = test_df['label'].iloc[mask.nonzero()[0]]
    m = tier_metrics(sub_labels)
    m['filter'] = f'Consensus ({tier_name})'
    consensus_rows.append(m)

# Compare to LightGBM alone at same tiers
for tier_name in ['Standard', 'High', 'Premium']:
    mask = test_probas['LightGBM'] >= cutoffs_per_model['LightGBM'][tier_name]
    sub_labels = test_df['label'].iloc[mask.nonzero()[0]]
    m = tier_metrics(sub_labels)
    m['filter'] = f'LightGBM-only ({tier_name})'
    consensus_rows.append(m)

consensus_df = pd.DataFrame(consensus_rows)
print('Consensus vs LightGBM-only on TEST (in-sample symbols):')
display(consensus_df[['filter', 'n', 'wr', 'pf', 'expR']].round(4))

# Same on GBPUSD hold-out
ho_consensus_rows = []
for tier_name in ['Standard', 'High', 'Premium']:
    mask = (
        (ho_test_probas['LightGBM'] >= cutoffs_per_model['LightGBM'][tier_name]) &
        (ho_test_probas['XGBoost']  >= cutoffs_per_model['XGBoost'][tier_name]) &
        (ho_test_probas['CatBoost'] >= cutoffs_per_model['CatBoost'][tier_name])
    )
    sub_labels = ho_df['label'].iloc[mask.nonzero()[0]]
    m = tier_metrics(sub_labels)
    m['filter'] = f'Consensus ({tier_name})'
    ho_consensus_rows.append(m)
for tier_name in ['Standard', 'High', 'Premium']:
    mask = ho_test_probas['LightGBM'] >= cutoffs_per_model['LightGBM'][tier_name]
    sub_labels = ho_df['label'].iloc[mask.nonzero()[0]]
    m = tier_metrics(sub_labels)
    m['filter'] = f'LightGBM-only ({tier_name})'
    ho_consensus_rows.append(m)
ho_cons_df = pd.DataFrame(ho_consensus_rows)
print('\nConsensus vs LightGBM-only on GBPUSD hold-out:')
display(ho_cons_df[['filter', 'n', 'wr', 'pf', 'expR']].round(4))


## 8. Pine-Export-Eignung

In [ ]:
exportability = {
    'LightGBM': {'pine_ready': True,  'notes': 'Native tree-cascade, well-understood, our baseline'},
    'XGBoost':  {'pine_ready': True,  'notes': 'Similar tree structure, exportable with similar effort'},
    'CatBoost': {'pine_ready': False, 'notes': 'Oblivious trees + categorical embeddings — complex/impossible Pine export'},
    'Voting':   {'pine_ready': True,  'notes': 'Need to embed 3 models in Pine — ~3x line count, possible but bulky'},
}
exp_df = pd.DataFrame(exportability).T
print('Pine export feasibility per model:')
display(exp_df)

## 9. Decision Logic

In [ ]:
print('=' * 70)
print('MODEL BATTERY VERDICT')
print('=' * 70)

premium_pfs = battery_df[battery_df['tier'] == 'Premium'].set_index('model')['pf'].to_dict()
lgbm_pf = premium_pfs['LightGBM']
print(f'\nLightGBM baseline Premium PF: {lgbm_pf:.4f}')
print()
for name in ['XGBoost', 'CatBoost', 'Voting']:
    pf = premium_pfs[name]
    lift = pf - lgbm_pf
    pine_ready = exportability[name]['pine_ready']
    ready_str = 'Pine-ready' if pine_ready else 'Backend-only'
    print(f'  {name:<10s} PF {pf:.4f}  lift {lift:+.4f}  [{ready_str}]')

best_name = max(premium_pfs, key=premium_pfs.get)
best_pf = premium_pfs[best_name]
best_lift = best_pf - lgbm_pf
best_pine = exportability[best_name]['pine_ready']

print('\n' + '-' * 70)
print(f'Best model: {best_name} (Premium PF {best_pf:.4f}, lift {best_lift:+.4f})')
if best_name == 'LightGBM' or best_lift < 0.05:
    print('-> LightGBM remains the choice (lift too small to justify complexity)')
elif best_pine:
    print(f'-> {best_name} wins AND is Pine-exportable')
else:
    print(f'-> {best_name} wins but is NOT Pine-exportable')
    print(f'   Decision tree:')
    print(f'   - If V1 must be Pine-only: use 2nd-best Pine-ready model')
    print(f'   - If accept Backend-V1: use {best_name} via webhook → TradingView')

print('=' * 70)

---

Schick mir nach dem Run:
- Section 6 (PF/WR/Expectancy pro Modell × Tier + AUC)
- Section 7 (Per-Year Stability Tabelle)
- Section 8 (Pine-Export-Eignung)
- Section 9 (Verdict)

Damit treffen wir die finale Architekturentscheidung vor V1.